# Session 3: Neural Networks

In this exercise, we will finetune parameters for a Neural Network build to detect a DDoS attack.

The function "ddos_detection_NN" build the Neural Network with the parameters in the cell below it.


In the **first part** of this Notebook called "Data" we will download and briefly inspect the dataset we will work with.

In the **second part** called "Neural Network Architecture & Parameters" we will build and train a basic neural network. Your goal is it to adjust the parameters and optimize for model accuracy.

In [ ]:
import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import time

import random

import os

from sklearn.preprocessing import LabelEncoder

from sklearn.model_selection import train_test_split

from sklearn.model_selection import StratifiedKFold

le = LabelEncoder()

# Data

In this chapter you don't need to edit anything, just download and briefly inspect the dataset we will be working with.

In [ ]:
url = "data/bccc-cpacket-cloud-ddos-2024-merged_small.csv"

data = pd.read_csv(url)

data.head()

This dataset contains network flow records with 319 features for each sample. Each row represents a single network flow, including details such as source and destination ports, flow duration, packet counts, payload statistics, and various statistical features for both forward and backward traffic directions. The dataset also includes categorical labels describing the flow’s nature, such as "Benign" or other activities.



For more infomration on the dataset, please refer to this [paper](https://www.mdpicom/2078-2489/15/4/195).

*Shafi, MohammadMoein, et al. "Toward generating a new cloud-based Distributed Denial of Service (DDoS) dataset and cloud intrusion traffic characterization." Information 15.4 (2024): 195.*

In [ ]:
data.fillna(value=np.nan, inplace=True)

data = data[(data.astype(str) != '?').all(axis=1)]

data = data.reset_index(drop = True)
# data = data.drop(columns=["flow_id", "timestamp", "src_ip", "dst_ip", "protocol"])
data.head()

In [ ]:
y = data.iloc[:,-2:-1]
y = y.where(y == "Benign", "Attack")
y.head()

In [ ]:
y = le.fit_transform(y)

In [ ]:
X = data.iloc[:,:-2]
X = np.array(X)


# Neural Network Architecture & Parameters

Here we build a neural network, train it with our data and evaluate it.

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn import metrics

##  Parameters - Edit here

Start by editing these values here. They will be used to build the Neural Network and affect the accuracy of the final model.




In [ ]:
# ====== NN Hyperparameters ======

# Network size 
HIDDEN_LAYERS = (16, 8, 4)  # Try: (64, 32, 16), (128, 64, 32), (256, 128, 64), etc.

# Activation function for hidden layers
ACTIVATION = 'relu'        # Try: 'relu', 'tanh', 'logistic' (sigmoid)

# L2 regularization
ALPHA = 0.001              # Try: 0.00001 to 0.001 (common 0.00001 to 0.0001)

# Optimizer
LEARNING_RATE_INIT = 0.000001  # Try: 0.000001 to 0.001

# Learning rate schedule
LEARNING_RATE_SCHEDULE = 'constant'  # Try: 'constant', 'invscaling', 'adaptive'

# Training process
BATCH_SIZE = 64            # Try: 8, 16, 32, 64, 128
MAX_ITER = 100             # Try: 50 to 300
N_ITER_NO_CHANGE = 5       # Try: 5 to 20 (early stopping patience)
TOL = 0.0001               # Try: 0.00001 to 0.00100

## Run

In [ ]:
def ddos_detection_NN(Xtrain, Xtest, ytrain, ytest):
    clf = MLPClassifier(
        hidden_layer_sizes=HIDDEN_LAYERS,
        activation=ACTIVATION,
        solver='adam',
        alpha=ALPHA,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE_SCHEDULE,
        learning_rate_init=LEARNING_RATE_INIT,
        max_iter=MAX_ITER,
        early_stopping=True,
        n_iter_no_change=N_ITER_NO_CHANGE,
        tol=TOL,
        validation_fraction=0.2,
        random_state=1337,
        verbose=False
    )
    clf.fit(Xtrain, ytrain)
    pred = clf.predict(Xtest)

    # Metrics
    accuracy = metrics.accuracy_score(ytest, pred)
    precision = metrics.precision_score(ytest, pred, average='weighted')
    recall = metrics.recall_score(ytest, pred, average='weighted')
    f1 = metrics.f1_score(ytest, pred, average='weighted')
    fbeta = metrics.fbeta_score(ytest, pred, average='weighted', beta=10)
    tn, fp, fn, tp = metrics.confusion_matrix(ytest, pred).ravel()
    tpr = tp / (tp + fn)
    fpr = fp / (fp + tn)

    # Plot training loss curve
    plt.plot(clf.loss_curve_, label='train')
    plt.title('Model Loss')
    plt.ylabel('loss')
    plt.xlabel('iteration')
    plt.legend(loc='upper right')
    plt.show()

    print(f"Accuracy:    {accuracy:.4f}   — Fraction of all predictions that are correct")
    print(f"Precision:   {precision:.4f}   — Of all predicted positives, how many are truly positive (low = many false alarms)")
    print(f"Recall:      {recall:.4f}   — Of all actual positives, how many were detected (low = missed attacks)")
    print(f"F1 Score:    {f1:.4f}   — Harmonic mean of precision and recall (balanced trade-off)")
    print(f"F-Beta (β=10): {fbeta:.4f} — Like F1, but weighs recall {10}x more than precision (prioritizes catching attacks)")
    print(f"TPR:         {tpr:.4f}   — True Positive Rate (same as recall; fraction of attacks detected)")
    print(f"FPR:         {fpr:.4f}   — False Positive Rate (fraction of benign traffic flagged as attacks)")
    print(f"\nConfusion Matrix:  TP={tp}  FP={fp}  FN={fn}  TN={tn}")
    print(f"Iterations:  {clf.n_iter_}  (out of max {MAX_ITER})")

    return fbeta, tpr, fpr

In [ ]:
# make the NN deterministic (for educational purposes)
os.environ['PYTHONHASHSEED'] = '1337'
random.seed(1337)
np.random.seed(1337)
#####################################################


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1337, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

start_time = time.time()

ddos_detection_NN(Xtrain=X_train, Xtest=X_test, ytrain=y_train, ytest=y_test)

print("\n--- %d minutes %d seconds ---" % (int((time.time() - start_time) // 60), int((time.time() - start_time) % 60)))